In [0]:
df = spark.read.table("creditcard")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
display(df.limit(5))

Rows: 284,807
Columns: 31


Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0.0,-1.3598071336738,-0.0727811733098497,2.53634673796914,1.37815522427443,-0.338320769942518,0.462387777762292,0.239598554061257,0.0986979012610507,0.363786969611213,0.0907941719789316,-0.551599533260813,-0.617800855762348,-0.991389847235408,-0.311169353699879,1.46817697209427,-0.470400525259478,0.207971241929242,0.0257905801985591,0.403992960255733,0.251412098239705,-0.018306777944153,0.277837575558899,-0.110473910188767,0.0669280749146731,0.128539358273528,-0.189114843888824,0.133558376740387,-0.0210530534538215,149.62,0
0.0,1.19185711131486,0.26615071205963,0.16648011335321,0.448154078460911,0.0600176492822243,-0.0823608088155687,-0.0788029833323113,0.0851016549148104,-0.255425128109186,-0.166974414004614,1.61272666105479,1.06523531137287,0.48909501589608,-0.143772296441519,0.635558093258208,0.463917041022171,-0.114804663102346,-0.183361270123994,-0.145783041325259,-0.0690831352230203,-0.225775248033138,-0.638671952771851,0.101288021253234,-0.339846475529127,0.167170404418143,0.125894532368176,-0.00898309914322813,0.0147241691924927,2.69,0
1.0,-1.35835406159823,-1.34016307473609,1.77320934263119,0.379779593034328,-0.503198133318193,1.80049938079263,0.791460956450422,0.247675786588991,-1.51465432260583,0.207642865216696,0.624501459424895,0.066083685268831,0.717292731410831,-0.165945922763554,2.34586494901581,-2.89008319444231,1.10996937869599,-0.121359313195888,-2.26185709530414,0.524979725224404,0.247998153469754,0.771679401917229,0.909412262347719,-0.689280956490685,-0.327641833735251,-0.139096571514147,-0.0553527940384261,-0.0597518405929204,378.66,0
1.0,-0.966271711572087,-0.185226008082898,1.79299333957872,-0.863291275036453,-0.0103088796030823,1.24720316752486,0.23760893977178,0.377435874652262,-1.38702406270197,-0.0549519224713749,-0.226487263835401,0.178228225877303,0.507756869957169,-0.28792374549456,-0.631418117709045,-1.0596472454325,-0.684092786345479,1.96577500349538,-1.2326219700892,-0.208037781160366,-0.108300452035545,0.00527359678253453,-0.190320518742841,-1.17557533186321,0.647376034602038,-0.221928844458407,0.0627228487293033,0.0614576285006353,123.5,0
2.0,-1.15823309349523,0.877736754848451,1.548717846511,0.403033933955121,-0.407193377311653,0.0959214624684256,0.592940745385545,-0.270532677192282,0.817739308235294,0.753074431976354,-0.822842877946363,0.53819555014995,1.3458515932154,-1.11966983471731,0.175121130008994,-0.451449182813529,-0.237033239362776,-0.0381947870352842,0.803486924960175,0.408542360392758,-0.00943069713232919,0.79827849458971,-0.137458079619063,0.141266983824769,-0.206009587619756,0.502292224181569,0.219422229513348,0.215153147499206,69.99,0


In [0]:
# Write Bronze Delta table - raw data, no changes
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("bronze_creditcard")

print("Bronze table written successfully")
print(f"Row count: {spark.read.table('bronze_creditcard').count():,}")

Bronze table written successfully
Row count: 284,807


In [0]:
from pyspark.sql.functions import col, sum as spark_sum, isnan, when, count

# Check for nulls across all columns
null_counts = df.select([
    spark_sum(when(col(c).isNull() | isnan(c), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(null_counts)

Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
from pyspark.sql.functions import col, when, hour, round as spark_round

# Silver layer - clean and typed
silver_df = spark.read.table("bronze_creditcard") \
    .withColumn("Amount", spark_round(col("Amount"), 2)) \
    .withColumn("Class", col("Class").cast("integer")) \
    .withColumn("is_fraud", when(col("Class") == 1, "Yes").otherwise("No")) \
    .withColumn("amount_bracket", 
        when(col("Amount") < 10, "micro")
        .when(col("Amount") < 100, "small")
        .when(col("Amount") < 1000, "medium")
        .otherwise("large")
    ) \
    .withColumn("time_of_day",
        when((col("Time") % 86400) < 21600, "night")
        .when((col("Time") % 86400) < 43200, "morning")
        .when((col("Time") % 86400) < 64800, "afternoon")
        .otherwise("evening")
    )

# Save Silver table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_creditcard")

print("Silver table written successfully")
display(silver_df.select("Time", "Amount", "amount_bracket", "time_of_day", "Class", "is_fraud").limit(10))

Silver table written successfully


Time,Amount,amount_bracket,time_of_day,Class,is_fraud
0.0,149.62,medium,night,0,No
0.0,2.69,micro,night,0,No
1.0,378.66,medium,night,0,No
1.0,123.5,medium,night,0,No
2.0,69.99,small,night,0,No
2.0,3.67,micro,night,0,No
4.0,4.99,micro,night,0,No
7.0,40.8,small,night,0,No
7.0,93.2,small,night,0,No
9.0,3.68,micro,night,0,No


In [0]:
from pyspark.sql.functions import col, avg, stddev, count, when, round as spark_round

# Read silver
silver = spark.read.table("silver_creditcard")

# Gold layer - feature engineered, analytics ready
gold_df = silver \
    .withColumn("amount_log", spark_round(col("Amount") + 1, 4)) \
    .withColumn("high_value_flag", when(col("Amount") > 1000, 1).otherwise(0)) \
    .withColumn("v1_v2_interaction", spark_round(col("V1") * col("V2"), 4)) \
    .withColumn("v3_v4_interaction", spark_round(col("V3") * col("V4"), 4))

# Save Gold table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_creditcard")

print("Gold table written successfully")

# Show class distribution - key insight
class_dist = gold_df.groupBy("is_fraud", "amount_bracket") \
    .count() \
    .orderBy("is_fraud", "amount_bracket")

display(class_dist)

Gold table written successfully


is_fraud,amount_bracket,count
No,large,3060
No,medium,54195
No,micro,97065
No,small,129995
Yes,large,9
Yes,medium,121
Yes,micro,249
Yes,small,113


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

# Load gold table
gold = spark.read.table("gold_creditcard")

# Define features (exclude label and non-numeric cols)
feature_cols = ["Time", "Amount", "amount_log", "high_value_flag",
                "v1_v2_interaction", "v3_v4_interaction"] + \
               [f"V{i}" for i in range(1, 29)]

# Assemble features into a single vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
assembled = assembler.transform(gold).select("features", col("Class").alias("label"))

# Train / test split
train, test = assembled.randomSplit([0.8, 0.2], seed=42)

# Train Gradient Boosted Tree classifier
gbt = GBTClassifier(labelCol="label", featuresCol="features", maxIter=20)
model = gbt.fit(train)

# Evaluate
predictions = model.transform(test)
evaluator = BinaryClassificationEvaluator(labelCol="label")
auc = evaluator.evaluate(predictions)
print(f"AUC-ROC: {auc:.4f}")

AUC-ROC: 0.9538


In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType

# UDF to extract fraud probability from sparse vector
extract_prob = udf(lambda v: float(v[1]), DoubleType())

# Build export table cleanly
pred_export = predictions.select(
    col("label"),
    col("prediction"),
    extract_prob(col("probability")).alias("fraud_probability")
)

# Add context columns from gold (no join needed - same order)
gold_select = spark.read.table("gold_creditcard").select(
    "Time", "Amount", "is_fraud", "amount_bracket", "time_of_day"
)

# Save separately and union by position
from pyspark.sql.functions import monotonically_increasing_id

pred_with_id = pred_export.withColumn("id", monotonically_increasing_id())
gold_with_id = gold_select.withColumn("id", monotonically_increasing_id())

final_export = pred_with_id.join(gold_with_id, on="id").drop("id").limit(50000)

final_export.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_predictions")

print("Predictions saved!")
display(final_export.limit(10))

Predictions saved!


label,prediction,fraud_probability,Time,Amount,is_fraud,amount_bracket,time_of_day
0,0.0,0.043668673786833034,0.0,149.62,No,medium,night
0,0.0,0.043668673786833034,0.0,2.69,No,micro,night
0,0.0,0.043668673786833034,1.0,378.66,No,medium,night
0,0.0,0.04376481229268292,1.0,123.5,No,medium,night
0,0.0,0.043668673786833034,2.0,69.99,No,small,night
0,0.0,0.043750407559447146,2.0,3.67,No,micro,night
0,0.0,0.043679877037982706,4.0,4.99,No,micro,night
0,0.0,0.043668673786833034,7.0,40.8,No,small,night
0,0.0,0.04368721320291369,7.0,93.2,No,small,night
0,0.0,0.043668673786833034,9.0,3.68,No,micro,night


In [0]:
# Convert to pandas and download directly - bypasses DBFS restriction
final_pandas = final_export.toPandas()

print(f"Rows exported: {len(final_pandas):,}")
print(f"Columns: {list(final_pandas.columns)}")

# Save as CSV in the notebook's local storage
final_pandas.to_csv("/tmp/fraud_predictions.csv", index=False)
print("CSV saved!")

Rows exported: 50,000
Columns: ['label', 'prediction', 'fraud_probability', 'Time', 'Amount', 'is_fraud', 'amount_bracket', 'time_of_day']
CSV saved!


In [0]:
# This creates a download link directly in the notebook
import base64

with open("/tmp/fraud_predictions.csv", "rb") as f:
    data = f.read()

b64 = base64.b64encode(data).decode()
download_link = f'<a href="data:text/csv;base64,{b64}" download="fraud_predictions.csv">Click here to download fraud_predictions.csv</a>'

displayHTML(download_link)

Click here to download fraud_predictions.csv